In [32]:
# -*- coding: utf-8 -*-
import os
#from pathlib import Path
#from typing import List, Optional
import pandas as pd
#import pyarrow as pya
import pyarrow.parquet as pq
#import pyarrow.dataset as ds
#import glob
#from itertools import chain
import numpy as np
from collections import deque, defaultdict
#from typing import Dict, List, Optional, Tuple
#import matplotlib.pyplot as plt
#from sklearn.metrics import accuracy_score, f1_score, classification_report
import os, urllib.parse, urllib.request
#import codecs
#from sklearn import metrics
#from scipy import interpolate

In [33]:
df1 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump1.parquet')
df2 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump2.parquet')
df3 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump3.parquet')
df4 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump4.parquet')
df5 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump5.parquet')
df6 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump6.parquet')
df7 = pq.read_table(r'C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\benign\dump7.parquet')

dump1 = df1.to_pandas()
dump2 = df2.to_pandas()
dump3 = df3.to_pandas()
dump4 = df4.to_pandas()
dump5 = df5.to_pandas()
dump6 = df6.to_pandas()
dump7 = df7.to_pandas()

In [34]:
files_pathname = r"C:\Users\User\Desktop\OneDrive - Universita degli Studi Roma Tre\Desktop\MATERIALE TESI POLIMI\Filtered Papers\To Study In Depth\Possible Pipeline\First Layer\New_strategyV2\New_strategy\X-CANIDS\thresholds_test"
all_files = [
    f for f in os.listdir(files_pathname)
    #if f.startswith("dump6-fuzz-") and f.endswith(".parquet")
]

all_ids = [f.split('-')[-1].removesuffix('.parquet') for f in all_files]

print(f"Found {len(all_ids)} files.")

Found 87 files.


In [35]:

def hex_to_bytes(h):
    #Convert hex/bytes/string to bytes.
    if h is None or (isinstance(h, float) and np.isnan(h)): 
        return b""
    if isinstance(h, (bytes, bytearray)): 
        return bytes(h)
    s = str(h).strip().replace("0x","").replace(" ","")
    if s == "": 
        return b""
    if len(s) % 2 == 1: 
        s = "0"+s
    try: 
        return bytes.fromhex(s)
    except Exception: 
        print(f"Warning: invalid hex input: {h}")
        return str(h).encode("utf-8", errors="ignore")


def hamming_distance(a: bytes, b: bytes) -> int:
    
    
    #len_mismatch = (len(a) != len(b))

    # Pad to same length
    max_len = max(len(a), len(b))
    a_padded = a + b'\x00' * (max_len - len(a))
    b_padded = b + b'\x00' * (max_len - len(b))
    
    # Count differences
    #dist = sum(byte_a != byte_b for byte_a, byte_b in zip(a_padded, b_padded))
    
    distance = 0
    for byte_a, byte_b in zip(a_padded, b_padded):
        distance += bin(byte_a ^ byte_b).count('1')
    return (distance)
    
    #return (dist, len_mismatch)

In [36]:
def compute_hamming_distances_testing(row, prev_payload, ranges_indexed):
    
    aid = row.arbitration_id
    lab = int(row.label) if hasattr(row, 'label') and pd.notna(row.label) else 0
    data = row.data
    curr_bytes = hex_to_bytes(data)
    
    # Get previous payload for this ID
    prev = prev_payload.get(aid)
    
    if prev is not None:
        # Compute Hamming distance
        dist = hamming_distance(curr_bytes, prev)
        #should_update = False
        
        """try:
            if ranges_indexed is not None and aid in ranges_indexed.index:
                min_val = ranges_indexed.at[aid, 'min_hamming']
                max_val = ranges_indexed.at[aid, 'max_hamming']
        except KeyError:
            pass
        
        if min_val is not None and max_val is not None:
            if min_val <= dist <= max_val:
                should_update = True
            #if outside range = suspicious, don't update
            else:
                #unknown ID (not in training) - don't update
                should_update = False
                
        if should_update: """  
        prev_payload[aid] = curr_bytes
        #ASK PROFESSOR - what to do if the previous payload is malicious because you compare a valid one with a malicious that you saved in the payload_prev and you get a FP????
        return dist, lab, prev_payload
    # Always update
    """else:
        # First message for this ID - store it
        prev_payload[aid] = curr_bytes
        return None, lab, prev_payload"""
        # Update previous payload
    prev_payload[aid] = curr_bytes
    return None, lab, prev_payload

In [37]:

def detect_simple(dist, arb_id, label, ranges_lookup):
    """
    Simple detection: Flag if distance is OUTSIDE [min, max] range.
    No windowing - direct per-message detection.
    
    Args:
        test_csv: Path to test data with Hamming distances
        ranges_csv: Path to learned ranges
        output_csv: Where to save results
        
    Returns:
        DataFrame with detection results and performance metrics
    """
    #check_anomaly_and_labels = []
    # Load data
    if arb_id not in ranges_lookup:
        return {
            'arbitration_id': arb_id,
            'hamming_distance': dist,
            'detected': True,  # Unknown ID = anomaly
            'label': label,
            'reason': 'unknown_id'
        }
    
    min_ham, max_ham = ranges_lookup[arb_id]
    detected = (dist < min_ham) or (dist > max_ham)
    
    return {
        'arbitration_id': arb_id,
        'hamming_distance': dist,
        'detected': detected,
        'label': label,
        'min_range': min_ham,
        'max_range': max_ham
    }

In [38]:
def analyze_packet_decision(detection_result, threshold_low, threshold_high):
    """
    Combines Score Calculation and Decision Making into one step.
    Replaces the logic previously found in 'prepare_roc_data' and 'compute_metrics'.
    """
    # --- 1. CALCULATE ANOMALY SCORE ---
    # Logic from prepare_roc_data_efficient
    if detection_result.get('reason') == 'unknown_id':
        anomaly_score = 100.0  # Max penalty for unknown IDs
    elif detection_result['detected']:
        dist = detection_result['hamming_distance']
        min_ham = detection_result['min_range']
        max_ham = detection_result['max_range']
        
        # How far outside the range?
        if dist < min_ham:
            anomaly_score = float(min_ham - dist)
        else: # dist > max_ham
            anomaly_score = float(dist - max_ham)
    else:
        anomaly_score = 0.0
        
    # --- 2. DETERMINE DECISION STAGE ---
    # Logic from compute_metrics_from_detections
    if anomaly_score >= threshold_high:
        decision = "high_confidence"  # Alert / Block
    elif anomaly_score >= threshold_low:
        decision = "low_confidence"   # Send to Second System
    else:
        decision = "normal"
        
    return anomaly_score, decision

In [ ]:
def compute_thesis_metrics(detections_df):
    """
    Calculates metrics applying the 'Two-Stage' logic AND the 'Collateral Filter'.
    
    Logic:
    1. Direct Detection: Only 'high_confidence' counts as a System Alert.
    2. Suspicious: 'low_confidence' packets are counted but not treated as immediate Alerts.
    3. Collateral Filter: We IGNORE False Positives if the previous packet was an attack.
       (Thesis argument: System knows it is in an unstable state).
       
       
    """
    
    detections_df['attack_time'] = detections_df['timestamp'].where(detections_df['label'] == 1)
    
    # Forward-fill (ffill) the attack times, strictly grouped by the CAN ID.
    # This ensures an attack on ID 'X' only creates a cooldown for future messages of ID 'X'.
    detections_df['last_attack_time'] = detections_df.groupby('arbitration_id')['attack_time'].ffill()
    
    # Smart Cooldown Setup (safely handles Timedelta vs Float):
    sample_ts = detections_df['timestamp'].iloc[0]
    if isinstance(sample_ts, (pd.Timedelta, pd.Timestamp)):
        COOLDOWN = pd.Timedelta(milliseconds=500.0)
    else:
        # Assuming typical float timestamps in CAN datasets are seconds (0.5s = 500ms)
        COOLDOWN = 0.5
    # ---------------------------------------------------------
    # 1. IDENTIFY SUBSETS (The Thesis Logic)
    # ---------------------------------------------------------
    
    # A) ATTACK SUBSET (For TPR)
    # Ground Truth = Attack
    attack_subset = detections_df[detections_df['label'] == 1]
    
    # B) CLEAN BENIGN SUBSET (For FPR)
    # Ground Truth = Benign AND Previous = Benign
    # This is your "Collateral Filter" - we strictly exclude dirty history
    no_previous_attack = detections_df['last_attack_time'].isna()
    cooldown_passed = detections_df['timestamp'] > (detections_df['last_attack_time'] + COOLDOWN)
    
    clean_benign_subset = detections_df[
        (detections_df['label'] == 0) & 
        (no_previous_attack | cooldown_passed)
    ]
    
    # C) COLLATERAL SUBSET (Just for reporting)
    # Ground Truth = Benign BUT Previous = Attack
    collateral_subset = detections_df[
        (detections_df['label'] == 0) & 
        (~no_previous_attack) & (~cooldown_passed)
    ]

    # ---------------------------------------------------------
    # 2. CALCULATE METRICS (Direct Detection / High Conf)
    # ---------------------------------------------------------
    
    # TP: Attacks detected immediately (High Confidence)
    tp = len(attack_subset[attack_subset['detection_stage'] == 'high_confidence'])
    
    # FN: Attacks MISSED by immediate check (Normal OR Low Confidence)
    # Note: Low Conf counts as FN here because the first stage didn't block it.
    fn = len(attack_subset[attack_subset['detection_stage'] != 'high_confidence'])
    
    # FP: Clean Benign wrongly alerted as High Confidence
    fp = len(clean_benign_subset[clean_benign_subset['detection_stage'] == 'high_confidence'])
    
    # TN: Clean Benign correctly passed (Normal OR Suspicious)
    # Note: If it's suspicious, we didn't Alert, so it's a True Negative for the "Alert System".
    tn = len(clean_benign_subset[clean_benign_subset['detection_stage'] != 'high_confidence'])
    
    # ---------------------------------------------------------
    # 3. SECOND SYSTEM STATS
    # ---------------------------------------------------------
    
    # How many packets (Attack OR Benign) were sent to the second system?
    n_suspicious = len(detections_df[detections_df['detection_stage'] == 'low_confidence'])
    
    # Optional: Breakdown of what the suspicious packets actually are
    suspicious_attacks = len(attack_subset[attack_subset['detection_stage'] == 'low_confidence'])
    
    # ---------------------------------------------------------
    # 4. RATES
    # ---------------------------------------------------------
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * tpr) / (precision + tpr) if (precision + tpr) > 0 else 0
    
    return {
        'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn,
        'TPR': tpr, 'FPR': fpr, 'F1': f1,
        'Suspicious_Total': n_suspicious,
        'Suspicious_Attacks': suspicious_attacks,
        'Collateral_Ignored': len(collateral_subset)
    }

In [40]:
def detect_nominal_periods(benign_dumps, candidate_periods=[10, 20, 100, 200, 1000, 2000]):
    """
    Automatically detect the nominal period for each CAN ID from benign data.
    
    Args:
        benign_dumps: List of (name, dataframe) tuples
        candidate_periods: Common nominal periods in ms (from paper)
    
    Returns:
        nominal_period_map: Dict {arb_id: detected_nominal_period}
    """
    
    # Collect all intervals for each ID
    intervals_per_id = defaultdict(list)
    
    for dump_name, dump in benign_dumps:
        print(f"Analyzing {dump_name}...")
        
        d = dump.copy()
        
        # Ensure timestamp is a column
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp":
                d = d.reset_index()
            else:
                d = d.reset_index().rename(columns={"index": "timestamp"})
        
        # Sort by timestamp
        d = d.sort_values('timestamp')
        
        # Track previous timestamp per ID
        prev_timestamp = {}
        
        for row in d.itertuples():
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            if curr_aid in prev_timestamp:
                # Calculate interval
                interval = curr_time - prev_timestamp[curr_aid]
                
                # Convert to milliseconds
                if isinstance(interval, pd.Timedelta):
                    interval_ms = interval.total_seconds() * 1000
                elif isinstance(interval, np.timedelta64):
                    interval_ms = float(interval / np.timedelta64(1, 'ms'))
                else:
                    interval_ms = float(interval)
                intervals_per_id[curr_aid].append(interval_ms)
            
            prev_timestamp[curr_aid] = curr_time
    
    # Determine nominal period for each ID
    nominal_period_map = {}
    
    print("\n" + "="*80)
    print("Detected Nominal Periods:")
    print("="*80)
    
    for arb_id, intervals in intervals_per_id.items():
        if len(intervals) < 100:
            print(f"{arb_id}: Insufficient data ({len(intervals)} samples)")
            continue
        
        intervals_array = np.array(intervals)
        
        # Filter out extreme outliers using 5th and 95th percentiles, i tried to do it also for threshold and detection but was not the best idea
        
        """lower_bound = np.percentile(intervals_array, 5)
        upper_bound = np.percentile(intervals_array, 95)
        filtered_intervals = intervals_array[
            (intervals_array >= lower_bound) & 
            (intervals_array <= upper_bound)
        ]"""
        
        # Use median (robust to outliers)
        median_interval = np.median(intervals_array)
        
        """# Method 2: Use mode (most common value)
        # Round to nearest ms to find mode
        intervals_rounded = np.round(intervals_array).astype(int)
        counts = np.bincount(intervals_rounded)
        mode_interval = np.argmax(counts)"""
        
        # Find closest candidate period
        #closest_candidate = min(candidate_periods, 
                               #key=lambda x: abs(x - median_interval))
        
        # Decide which to use:
        # If median is close to a candidate period (within 10%), use candidate
        # Otherwise, use the actual median
        # NEW CODE (always use actual median):
        nominal_period = median_interval  # ← Use actual median, no snapping!
        #method = "median"
        
        nominal_period_map[arb_id] = nominal_period
        
        # Print statistics
        print(f"\n{arb_id}:")
        print(f"  Total samples: {len(intervals)}")
        print(f"  Filtered samples: {len(intervals_array)} (removed {len(intervals) - len(intervals_array)})")
        print(f"  Median (filtered): {median_interval:.2f} ms")
        print(f"  Mean (filtered): {np.mean(intervals_array):.2f} ms")
        print(f"  Std Dev (filtered): {np.std(intervals_array):.2f} ms")
        print(f"  Range (filtered): [{np.min(intervals_array):.2f}, {np.max(intervals_array):.2f}]")
        print(f"  → Selected nominal period: {nominal_period:.2f} ms")
            
    return nominal_period_map

In [41]:
def parse_dbc_for_pe_ids(dbc_file_path):
    """
    Parses the specific Hyundai DBC file format to identify PE messages.
    Looks for message names containing '_PE_' or 'Warning'.
    """
    pe_ids = set()
    if not os.path.exists(dbc_file_path):
        print(f"Warning: DBC file not found at {dbc_file_path}. Using statistical fallback only.")
        return pe_ids

    try:
        with open(dbc_file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                if line.startswith("BO_ "):
                    # Format: BO_ 1491 HU_DATC_PE_00: 8 CLU
                    parts = line.split()
                    # parts[0]=BO_, parts[1]=ID, parts[2]=Name
                    
                    try:
                        msg_id = int(parts[1])
                        msg_name = parts[2]
                        
                        # Heuristic based on your DBC content
                        if "_PE_" in msg_name or "Warning" in msg_name:
                            pe_ids.add(msg_id)
                    except:
                        continue
                        
        print(f"DBC ANALYSIS: Found {len(pe_ids)} Explicit PE IDs: {pe_ids}")
        return pe_ids
    except Exception as e:
        print(f"Error parsing DBC: {e}")
        return set()

In [42]:
def compute_thresholds(benign_dumps, nominal_period_map, K, pe_dbc_list):
    print("COMPUTING THRESHOLDS (Hybrid DBC + Statistical Mode)")
    
    # 1. GET DBC KNOWLEDGE
    dbc_pe_ids = pe_dbc_list
    
    # 2. LOAD ALL RAW DATA (No filtering yet)
    residuals_per_id = defaultdict(list)
    
    for dump_name, dump in benign_dumps:
        print(f"Processing {dump_name}...")
        d = dump.copy()
        
        if "timestamp" not in d.columns:
            if d.index.name == "timestamp": d = d.reset_index()
        d = d.sort_values('timestamp')
        
        prev_timestamp = {}
        
        for row in d.itertuples():
            curr_time = row.timestamp
            curr_aid = row.arbitration_id
            
            nominal = nominal_period_map.get(curr_aid)
            if nominal is None: continue
                
            if curr_aid in prev_timestamp:
                time_diff = curr_time - prev_timestamp[curr_aid]
                if isinstance(time_diff, pd.Timedelta):
                    diff_ms = time_diff.total_seconds() * 1000
                else:
                    diff_ms = float(time_diff)
                
                # Store everything. We classify later.
                residual = diff_ms - nominal
                residuals_per_id[curr_aid].append(residual)
                    
            prev_timestamp[curr_aid] = curr_time

    # 3. CLASSIFY AND COMPUTE
    thresholds_per_id = {}
    
    print(f"\n{'ID':<6} {'Source':<10} {'Raw σ':<10} {'Final σ':<10} {'Mode':<10}")
    print("-" * 60)

    for arb_id, residuals in residuals_per_id.items():
        if len(residuals) < 10: continue
        
        arr = np.array(residuals)
        nominal = nominal_period_map[arb_id]
        
        # --- CLASSIFICATION LOGIC ---
        
        # Check 1: Is it in the DBC list?
        is_dbc_pe = arb_id in dbc_pe_ids
        
        # Check 2: Is it Statistically Noisy? (Raw Sigma > 1.0ms)
        # This catches IDs that aren't named "PE" but behave like it (e.g., ID 68)
        raw_sigma = np.std(arr)
        is_stat_pe = raw_sigma > 1.0
        
        IS_PE = is_dbc_pe or is_stat_pe
        
        if IS_PE:
            # --- CASE A: PE / Noisy ---
            # Strategy: LOOSE. Accept the noise.
            # We only filter extreme artifacts (> 5 sigma) to prevent 
            # dataset glitches (like packet drops) from ruining the mean.
            # We DO NOT filter fast packets.
            
            mu_temp = np.mean(arr)
            # Simple Sigma Clip (Wide)
            clean_arr = arr[abs(arr - mu_temp) < 5 * raw_sigma]
            
            source_str = "DBC" if is_dbc_pe else "Stats"
            mode_str = "LOOSE"
            
        else:
            # --- CASE B: Periodic / Stable ---
            # Strategy: STRICT.
            # This ID is supposed to be stable. Any burst is likely a payload-based event
            # that we cannot parse (missing payload algo), OR a bus artifact.
            # We MUST filter these to get a tight threshold for Masquerade detection.
            
            # Reconstruct interval
            intervals = arr + nominal
            
            # Filter 1: Impossible Speed (< 50% nominal)
            mask_fast = intervals >= (nominal * 0.5)
            
            # Filter 2: Massive Gap (> 3x nominal)
            mask_slow = intervals <= (nominal * 3.0)
            
            clean_arr = arr[mask_fast & mask_slow]
            
            source_str = "Periodic"
            mode_str = "STRICT"

        # Fallback if cleaning removed everything (rare)
        if len(clean_arr) == 0: 
            clean_arr = arr
            
        # Final Calculation
        mu = np.mean(clean_arr)
        sigma = np.std(clean_arr)
        
        thr_upper = K * sigma + mu
        thr_lower = -K * sigma + mu
        
        thresholds_per_id[arb_id] = {
            'lower': thr_lower,
            'upper': thr_upper,
            'mu': mu,
            'sigma': sigma,
            'K': K,
            'nominal_period': nominal
        }
        
        print(f"{arb_id:<6} {source_str:<10} {raw_sigma:<10.4f} {sigma:<10.4f} {mode_str:<10}")

    return thresholds_per_id

In [43]:
def detect_attacks_second_system(packet_list, nominal_period_map, thresholds_per_id):
    """
    SECOND SYSTEM DETECTION (Refactored for List Input).
    
    Accepts:
        packet_list: List of dicts [{'timestamp': t, 'id': aid, 'label': l, ...}, ...]
        nominal_period_map: {id: period_ms}
        thresholds_per_id: {id: {'lower': x, 'upper': y}}
        
    Combines:
    1. Saturation State Machine (Handles Delays/Congestion)
    2. History Protection (Handles History Pollution)
    """
    
    # PARAMETERS
    GLOBAL_WINDOW_SIZE = 50      
    SATURATION_THRESHOLD_MS = 40.0  
    COOLDOWN_MS = 100.0             

    # State
    prev_timestamp = {}          
    global_window = deque(maxlen=GLOBAL_WINDOW_SIZE)
    saturation_expiry_time = -1.0
    
    # Metrics
    total_benign = 0
    tp_packets = 0
    fn_packets = 0

    # Counters
    fp_clean = 0        # Algorithmic Error
    fp_suppressed = 0   # Collateral Delay (Correctly Ignored)
    fp_unexplained = 0  # True FP

    # TP ID tracking
    tp_ids = set()
    attacked_ids_gt = set()

    # --- PROCESS PACKET LIST ---
    # We iterate directly over the list passed from the First System
    for pkt in packet_list:
        
        # 1. Unpack Packet Data
        # Ensure timestamp is float (ms)
        raw_t = pkt['timestamp']
        try:
            if hasattr(raw_t, 'total_seconds'): 
                curr_time = raw_t.total_seconds() * 1000.0
            elif hasattr(raw_t, 'timestamp'):   
                curr_time = raw_t.timestamp() * 1000.0
            else:                               
                curr_time = float(raw_t) * 1000.0
        except: 
            curr_time = float(raw_t)

        curr_aid = pkt['id']
        # Default label to 0 if missing (assume benign if unknown)
        is_attack = (pkt.get('label', 0) == 1)

        if is_attack: attacked_ids_gt.add(curr_aid)
        
        # --- 2. GLOBAL MONITOR (SATURATION) ---
        global_window.append(curr_time)
        is_bus_saturated = False

        if len(global_window) == GLOBAL_WINDOW_SIZE:
            win_diff = global_window[-1] - global_window[0]
            # Robust Time Conversion
            if hasattr(win_diff, 'total_seconds'):
                val = win_diff.total_seconds() * 1000
            else:
                val = float(win_diff) * 1000

            if val < SATURATION_THRESHOLD_MS:
                # Bus is screaming. Set expiry
                saturation_expiry_time = curr_time + COOLDOWN_MS

        if saturation_expiry_time != -1.0:
            if curr_time < saturation_expiry_time:
                is_bus_saturated = True

        # --- 3. DETECTION LOGIC ---
        nominal = nominal_period_map.get(curr_aid)
        thr = thresholds_per_id.get(curr_aid)

        if not nominal or not thr: 
            tp_packets += 1
            tp_ids.add(curr_aid)
            continue
        
        # Initialize History if unseen
        if curr_aid not in prev_timestamp:
            prev_timestamp[curr_aid] = curr_time
            continue
        
        # Calculate Time Delta
        diff = curr_time - prev_timestamp[curr_aid]
        if hasattr(diff, 'total_seconds'):
            diff_ms = diff.total_seconds() * 1000
        else:
            diff_ms = float(diff) * 1000

        gradient = diff_ms - nominal
        is_too_fast = (gradient < thr['lower'])
        is_too_slow = (gradient > thr['upper'])

        # --- 4. CLASSIFICATION & HISTORY PROTECTION ---
        if is_attack:
            # TP Logic
            if is_too_fast or is_too_slow:
                tp_packets += 1
                tp_ids.add(curr_aid)
            else:
                fn_packets += 1

            # CRITICAL: HISTORY PROTECTION
            # If we detect an injection (too fast/slow), DO NOT update timestamp.
            if is_too_fast or is_too_slow:
                continue

        else:
            # Benign (Potential FP)
            total_benign += 1

            if is_too_fast:
                fp_clean += 1  # Algorithmic Error
                prev_timestamp[curr_aid] = curr_time

            elif is_too_slow:
                if is_bus_saturated:
                    fp_suppressed += 1 # Collateral (Ignored)
                else:
                    fp_unexplained += 1 # Unexplained Delay
                prev_timestamp[curr_aid] = curr_time
            else:
                # Normal Benign
                prev_timestamp[curr_aid] = curr_time

    # --- METRICS CALCULATION ---
    tpr_id = len(tp_ids) / len(attacked_ids_gt) if attacked_ids_gt else 0
    fpr_raw = (fp_clean + fp_unexplained + fp_suppressed) / total_benign if total_benign else 0
    fpr_clean = (fp_clean + fp_unexplained) / total_benign if total_benign else 0
    fp_total_raw = fp_clean + fp_unexplained + fp_suppressed

    # Return structured results
    return {
        'TPR_ID': tpr_id,
        'FPR_Raw': fpr_raw,
        'FPR_Clean': fpr_clean,
        'Suppressed': fp_suppressed,
        'FP_Algorithmic': fp_clean,
        'FP_Unexplained': fp_unexplained,
        'TP_PKT': tp_packets,
        'FN_PKT': fn_packets,
        'TN_PKT': total_benign - fp_total_raw,
        'FP_Total_PKT': fp_total_raw,
        'FP_Clean_PKT': fp_clean + fp_unexplained,
        'Suspicious_Attacks_Caught': tp_packets # Renamed for clarity in main loop
    }

In [44]:
#they say to compute onlty from idling data but dump6 which is the one with the attacks is in motion, i suppose i will use everything in motion then
benign_dumps = [
    ("dump1", dump1),
    ("dump2", dump2),
    ("dump3", dump3),
    ("dump4", dump4),
    ("dump5", dump5),
    ("dump6", dump6),
    #("dump7", dump7)
]


nominal_period_map = detect_nominal_periods(benign_dumps)
#frames = [dump1, dump2, dump3, dump4, dump5, dump6, dump7]

cumulative_residuals_per_id = defaultdict(list)
#nominal_periods = [10, 20 ,100, 200, 1000] # nominal periods in ms
#fuzz_intensity = ["10", "20", "30", "40", "50", "60", "70", "80", "90", "100", "200", "300", "400", "500", "1000", "1500", "2000"]

pe_list = parse_dbc_for_pe_ids('X-CANIDS/hyundai_2015_ccan.dbc')
K=5

print(f"\n{'='*80}")
print("COMPUTING THRESHOLDS")   
#TO-DO for each cumulative residual i have to calculate mean and std deviation to set thresholds, have to understand why better        
thresholds_per_id = compute_thresholds(benign_dumps, nominal_period_map, K, pe_list)    

thresholds_df = pd.DataFrame.from_dict(thresholds_per_id, orient='index')
thresholds_df.to_csv('ErrIDS_artifacts/thresholds.csv')


THRESH_LOW = 1.0
THRESH_HIGH = 10.0

print("\n" + "="*70)
print("PHASE 2: EVALUATION ON UNSEEN TEST DATA")
print("="*70)

global_benign_max_score = 0
global_benign_score_counts = pd.Series(dtype=int)
total_benign_packets_seen = 0

test_folder = os.path.join(files_pathname, "")
test_files = [f for f in os.listdir(files_pathname) 
            if f.startswith("dump6-") and f.endswith(".parquet")]


ranges_df = pd.read_csv("artifacts/hamming_ranges_universal.csv")
    # ========================================================================
ranges_indexed = ranges_df.set_index('arbitration_id')
    #results_summary = []
all_detections = []
    #i made this dict to speed up the lookup of ranges instead of querying the dataframe each time cause it took 4h per file 
ranges_lookup = {
    row['arbitration_id']: (row['min_hamming'], row['max_hamming'])
    for _, row in ranges_df.iterrows()
}

print(f"\nFound {len(test_files)} test files")

thesis_metrics = {
    'tp': 0, 
    'fn': 0, 
    'tn': 0, 
    'fp_pure': 0,      # The number you want to minimize
    'collateral': 0    # The number you explain away in text
}

# Track metrics incrementally
total_test_packets = 0
total_test_attacks = 0
total_test_normal = 0
test_by_type = {'fuzz': 0, 'masq': 0, 'fabr': 0}

# Metrics accumulators
total_tp = 0
total_tp_high = 0 
total_tp_low = 0  
total_fp = 0
total_tn = 0
total_fn = 0


# >>> NEW GLOBALS FOR SECOND SYSTEM <<<
total_second_tp = 0         # Attacks caught by the Timing/Logic check
total_second_fp = 0         # FPs created by the Timing/Logic check
total_second_suppressed = 0 # FPs avoided because of the Shockwave filter
total_suspicious_count = 0  # Total packets sent to Second System

type_metrics = {
'fuzz': {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0},
'masq': {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0},
'fabr': {'tp': 0, 'fp': 0, 'tn': 0, 'fn': 0}
}

# Process files one at a time
for i, filename in enumerate(all_files, 1):
    print(f"\n[TEST {i}/{len(all_files)}] Processing: {filename}")
    
    
    # Load file
    test_file_path = os.path.join(test_folder, filename)
    test_df = pq.read_table(test_file_path).to_pandas()
    
    print(f"   Loaded: {len(test_df):,} messages")
    
    # Detection
    prev_payload = {}
    test_dump = test_df.copy()
    
    if "timestamp" not in test_dump.columns:
        if test_dump.index.name == "timestamp":
            test_dump = test_dump.reset_index()
        else:
            test_dump = test_dump.reset_index().rename(columns={"index": "timestamp"})
    
    test_dump = test_dump.sort_values("timestamp", kind="mergesort")
    
    detect_list = []
    
    suspicious_series = [] # Buffer for Second System
    
    # Context Trackers
    prev_payload = {}
    prev_label_tracker = {} # Essential for 0% FPR logic
    
    for row in test_dump.itertuples():
        current_aid = row.arbitration_id
        test_ham_dist, lab, prev_payload = compute_hamming_distances_testing(
            row, 
            prev_payload,
            ranges_indexed
        )
        
        # 2. Get Previous Label (for Collateral Filter)
        previous_label_for_id = prev_label_tracker.get(current_aid, 0)
        
        if test_ham_dist is not None:
            results = detect_simple(
                test_ham_dist,
                current_aid,
                lab,  
                ranges_lookup
            )
            
            
            # 4. FULL ANALYSIS (The new function)
            score, decision = analyze_packet_decision(results, THRESH_LOW, THRESH_HIGH)
            
            # 5. ACTION TAKING
            if decision == "low_confidence":
                # >>> TRIGGER SECOND SYSTEM <<<
                suspicious_packet = {
                    'timestamp': row.timestamp,
                    'id': current_aid,
                    'score': score,
                    'data': row.data
                }
                suspicious_series.append(suspicious_packet)
                
            elif decision == "high_confidence":
                # >>> TRIGGER ALERT <<<
                pass

            # 6. SAVE RESULTS (No post-processing needed anymore!)
            results['timestamp'] = row.timestamp
            results['anomaly_score'] = score
            results['detection_stage'] = decision
            results['prev_label'] = previous_label_for_id
            results['label'] = lab # Ensure ground truth is saved
            
            detect_list.append(results)
            
            #Update the tracker for the next packet!
            prev_label_tracker[current_aid] = lab  
        
    # Convert to DataFrame
    detections_df = pd.DataFrame(detect_list)
    
  
    second_system_results = detect_attacks_second_system(suspicious_series, nominal_period_map, thresholds_per_id)
    
    print(f"   [SECOND SYSTEM]    Analyzed {len(suspicious_series)} packets.")
    print(f"                      Caught {second_system_results['TP_PKT']} attacks.")
    print(f"                      Suppressed {second_system_results['Suppressed']} collateral alarms.")
    
    
    # >>> ADD THIS UPDATE BLOCK <<<
    total_second_tp += second_system_results['TP_PKT']
    total_second_fp += second_system_results['FP_Clean_PKT']
    total_second_suppressed += second_system_results['Suppressed']
    total_suspicious_count += len(suspicious_series)
    
    # 2. CALL THE  METRICS FUNCTION
    metrics = compute_thesis_metrics(detections_df)
    
    # 
    # Calculate Comprehensive TPR (Direct + Second System)
    total_attacks = metrics['TP'] + metrics['FN']
    total_detected = metrics['TP'] + metrics['Suspicious_Attacks']
    tpr_comprehensive = total_detected / total_attacks if total_attacks > 0 else 0
    
    # 3. PRINT RESULTS
    print(f"   [DIRECT DETECTION] TPR: {metrics['TPR']:.2%} | FPR: {metrics['FPR']:.4%} | F1: {metrics['F1']:.4f}")
    print(f"   [COMPREHENSIVE]    TPR: {tpr_comprehensive:.2%} (Direct + Second System)")
    print(f"   [SECOND SYSTEM]    Suspicious Packets: {metrics['Suspicious_Total']:,} (Caught {metrics['Suspicious_Attacks']:,} attacks)")
    print(f"   [COLLATERAL]       Ignored {metrics['Collateral_Ignored']:,} benign packets due to shockwave.")

    # 4. UPDATE GLOBALS
    total_tp += metrics['TP']
    total_fp += metrics['FP']
    total_tn += metrics['TN']
    total_fn += metrics['FN']
    
    


# ========================================================================
# FINAL COMPREHENSIVE SUMMARY
# ========================================================================

print("\n" + "="*80)
print("FINAL THESIS PIPELINE RESULTS (DIRECT + SECOND SYSTEM)")
print("="*80)

# 1. Total Counts
# Total Attacks = Direct TP + Direct FN (High Conf hits + Misses)
total_attacks_global = total_tp + total_fn 
# Total Benign = Direct FP + Direct TN (High Conf False Alarms + Correct Rejections)
total_benign_global = total_fp + total_tn

# 2. Comprehensive Metrics
# Total Detected = Direct Hits + Second System Hits
total_detected_combined = total_tp + total_second_tp

# Total False Positives = Direct False Alarms + Second System False Alarms
# (Note: Direct TN included the suspicious packets. If 2nd system flags them wrong, they become FPs)
total_fp_combined = total_fp + total_second_fp

# Final Rates
tpr_final = total_detected_combined / total_attacks_global if total_attacks_global > 0 else 0
fpr_final = total_fp_combined / total_benign_global if total_benign_global > 0 else 0

# Precision / F1
precision_final = total_detected_combined / (total_detected_combined + total_fp_combined) if (total_detected_combined + total_fp_combined) > 0 else 0
f1_final = 2 * (precision_final * tpr_final) / (precision_final + tpr_final) if (precision_final + tpr_final) > 0 else 0

print(f"\nGLOBAL DATASET STATS:")
print(f"  Total Files Processed: {len(all_files)}")
print(f"  Total Attacks:         {total_attacks_global:,}")
print(f"  Total Benign:          {total_benign_global:,}")

print(f"\n" + "-"*40)
print(f"STAGE 1: DIRECT DETECTION (High Confidence)")
print(f"-"*40)
tpr_direct = total_tp / total_attacks_global if total_attacks_global > 0 else 0
fpr_direct = total_fp / total_benign_global if total_benign_global > 0 else 0

print(f"  TPR (Direct):          {tpr_direct:.2%}")
print(f"  FPR (Direct):          {fpr_direct:.4%} (High Confidence Alerts)")
print(f"  Attacks Caught:        {total_tp:,}")
print(f"  Attacks Missed:        {total_fn:,} (Sent to Stage 2 or Lost)")

print(f"\n" + "-"*40)
print(f"STAGE 2: SECOND SYSTEM (Timing & Logic)")
print(f"-"*40)
# Efficacy: Of the attacks sent to Stage 2, how many did we catch?
# We estimate 'Attacks Sent to Stage 2' as (Total Attacks - Direct TP)
attacks_in_stage_2 = total_attacks_global - total_tp
stage_2_efficacy = total_second_tp / attacks_in_stage_2 if attacks_in_stage_2 > 0 else 0

print(f"  Packets Analyzed:      {total_suspicious_count:,}")
print(f"  Attacks Recovered:     {total_second_tp:,} (rescued from misses)")
print(f"  Stage 2 Efficacy:      {stage_2_efficacy:.2%} (of attacks reaching this stage)")
print(f"  False Positives Added: {total_second_fp:,}")
print(f"  Collateral Suppressed: {total_second_suppressed:,} (FPs avoided by Shockwave filter)")

print(f"\n" + "="*80)
print(f"OVERALL SYSTEM PERFORMANCE")
print(f"="*80)
print(f"  COMPREHENSIVE TPR:     {tpr_final:.2%}  (Target: High)")
print(f"  COMPREHENSIVE FPR:     {fpr_final:.4%}  (Target: ~0%)")
print(f"  OVERALL F1 SCORE:      {f1_final:.4f}")
print(f"="*80)

Analyzing dump1...
Analyzing dump2...
Analyzing dump3...
Analyzing dump4...
Analyzing dump5...
Analyzing dump6...

Detected Nominal Periods:

356:
  Total samples: 1028843
  Filtered samples: 1028843 (removed 0)
  Median (filtered): 10.00 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.20 ms
  Range (filtered): [7.86, 79.97]
  → Selected nominal period: 10.00 ms

688:
  Total samples: 1028706
  Filtered samples: 1028706 (removed 0)
  Median (filtered): 10.00 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.63 ms
  Range (filtered): [6.22, 21.27]
  → Selected nominal period: 10.00 ms

593:
  Total samples: 1028705
  Filtered samples: 1028705 (removed 0)
  Median (filtered): 10.00 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.30 ms
  Range (filtered): [6.87, 20.38]
  → Selected nominal period: 10.00 ms

790:
  Total samples: 1028863
  Filtered samples: 1028863 (removed 0)
  Median (filtered): 9.99 ms
  Mean (filtered): 10.00 ms
  Std Dev (filtered): 0.85 ms
  Range 